# Day 004 — Stack · Shell Sort · Sentinel Search
**Date:** 2026-08-06  |  **Difficulty:** Beginner-Intermediate  |  **Series:** Daily DSA

---

## What You Will Learn Today
| Topic | Concept | Time Complexity |
|-------|---------|----------------|
| Data Structure | **Stack** | Push/Pop/Peek all O(1) |
| Sorting | **Shell Sort** | Best O(n log n), Worst O(n²) — gap-sequence dependent |
| Searching | **Sentinel Linear Search** | O(n) — fewer comparisons per iteration than plain Linear Search |

> **Series recap**
> - Day 001: Array · Bubble Sort · Linear Search O(n)
> - Day 002: Singly Linked List · Selection Sort · Jump Search O(√n)
> - Day 003: Doubly Linked List · Insertion Sort · Binary Search O(log n)
> - **Today**: Stack (LIFO) · Shell Sort (gap-sequence Insertion Sort) · Sentinel Search

> Run each cell top-to-bottom with **Shift+Enter** to follow along interactively.

---
## PART 1 — Data Structure: Stack

A **Stack** is a **Last-In, First-Out (LIFO)** collection.  
Think of a stack of plates — you always add and remove from the **top**.

### Visual
```
      ┌───────┐  ← top (most recently pushed)
      │   C   │
      ├───────┤
      │   B   │
      ├───────┤
      │   A   │  ← bottom (first pushed)
      └───────┘

push(D) → adds D on top    pop() → removes C from top
```

### Core operations — all O(1)
| Operation | Description |
|-----------|-------------|
| `push(x)` | Add element to the top |
| `pop()` | Remove and return the top element |
| `peek()` | Read the top element without removing it |
| `is_empty()` | True if the stack has no elements |

### Real-world uses
- **Function call stack** — the OS uses a stack to track nested function calls
- **Undo/Redo** — every text editor pushes actions onto a stack
- **Expression evaluation** — calculators use stacks to handle operator precedence
- **Balanced parentheses** — classic interview problem solved with a stack
- **Browser back-button** — visited pages pushed onto a stack

In [ ]:
# ─── Stack Implementation ─────────────────────────────────────────────────────────────────────────────

class Stack:
    """
    LIFO Stack backed by a Python list.
    All operations are O(1) amortised.

    Implementation note: Python's list.append() / list.pop() both
    operate on the END of the list, which is O(1) amortised —
    making it a perfect, zero-overhead stack backend.
    """

    def __init__(self, max_size=None):
        self._data = []
        self._max  = max_size   # optional capacity limit

    def push(self, item):
        """Add item to the top — O(1) amortised."""
        if self._max is not None and len(self._data) >= self._max:
            raise OverflowError(f'Stack is full (max_size={self._max})')
        self._data.append(item)

    def pop(self):
        """Remove and return the top item — O(1)."""
        if self.is_empty():
            raise IndexError('pop from empty stack')
        return self._data.pop()

    def peek(self):
        """Return the top item without removing it — O(1)."""
        if self.is_empty():
            raise IndexError('peek at empty stack')
        return self._data[-1]

    def is_empty(self):  return len(self._data) == 0
    def size(self):      return len(self._data)

    def __repr__(self):
        if self.is_empty():
            return 'Stack(empty)'
        items = ' | '.join(str(x) for x in self._data)
        return f'Stack(bottom [{items}] top)'


s = Stack()
print("--- Pushing A, B, C ---")
for ch in ['A', 'B', 'C']:
    s.push(ch)
    print(f"  push({ch!r}) → {s}")

print(f"\npeek()   → {s.peek()}  (stack unchanged: {s})")
print(f"size()   → {s.size()}")

print("\n--- Popping ---")
while not s.is_empty():
    print(f"  pop() → {s.pop()!r}  | stack: {s}")

In [ ]:
def is_balanced(expression):
    matching = {')': '(', ']': '[', '}': '{'}
    stack = Stack()
    for ch in expression:
        if ch in '([{':
            stack.push(ch)
        elif ch in ')]}' :
            if stack.is_empty() or stack.pop() != matching[ch]:
                return False
    return stack.is_empty()

tests = [
    ("(a + b) * [c - {d / e}]", True),
    ("((()))",                   True),
    ("([)]",                     False),
    ("{[}]",                     False),
    ("(",                        False),
    ("",                         True),
    ("print(arr[i])",            True),
    ("def f(): return {1: [2]}", True),
]

print(f"{'Expression':<40} {'Expected':>8}  {'Got':>5}  {'Pass'}")
print("-" * 65)
for expr, expected in tests:
    result = is_balanced(expr)
    ok = '✓' if result == expected else '✗'
    print(f"{repr(expr):<40} {str(expected):>8}  {str(result):>5}  {ok}")

---
## PART 2 — Sorting Algorithm: Shell Sort

Shell Sort is a **generalization of Insertion Sort** that first sorts elements **far apart**, then progressively reduces the gap until it reaches 1.

| Gap Sequence | Worst-case | Example gaps for n=100 |
|-------------|-----------|------------------------|
| Shell (n/2) | O(n²) | 50, 25, 12, 6, 3, 1 |
| Knuth (3k+1) | O(n^1.5) | 40, 13, 4, 1 |
| Ciura | O(n^?) best known | 57, 23, 10, 4, 1 |

In [ ]:
def shell_sort(arr, gap_sequence='shell', verbose=False):
    a = arr[:]
    n = len(a)
    total_shifts = 0
    if gap_sequence == 'knuth':
        gap = 1
        while gap < n // 3:
            gap = gap * 3 + 1
        gaps = []
        while gap >= 1:
            gaps.append(gap)
            gap //= 3
    else:
        gaps = []
        gap = n // 2
        while gap > 0:
            gaps.append(gap)
            gap //= 2
    for gap in gaps:
        pass_shifts = 0
        for i in range(gap, n):
            key = a[i]
            j = i
            while j >= gap and a[j - gap] > key:
                a[j] = a[j - gap]
                j -= gap
                pass_shifts += 1
                total_shifts += 1
            a[j] = key
        if verbose:
            print(f"  gap={gap:>3}: {pass_shifts:>3} shifts → {a}")
    return a, total_shifts, gaps

In [ ]:
sample = [12, 34, 54, 2, 3, 18, 7, 45]
print(f"Input : {sample}\n")
print("Shell gap sequence (n//2):")
sorted_shell, shifts_s, gaps_s = shell_sort(sample, gap_sequence='shell', verbose=True)
print(f"Result: {sorted_shell}  ({shifts_s} total shifts, gaps: {gaps_s})")
print("\nKnuth gap sequence (3k+1):")
sorted_knuth, shifts_k, gaps_k = shell_sort(sample, gap_sequence='knuth', verbose=True)
print(f"Result: {sorted_knuth}  ({shifts_k} total shifts, gaps: {gaps_k})")
print(f"\nKnuth saved {shifts_s - shifts_k} shifts vs Shell on this input.")

In [ ]:
test_cases = [
    ([12, 34, 54, 2, 3],     "random small"),
    ([1, 2, 3, 4, 5],        "already sorted"),
    ([5, 4, 3, 2, 1],        "reverse sorted"),
    ([42],                   "single element"),
    ([],                     "empty list"),
    ([3, 3, 1, 1, 2],        "with duplicates"),
    ([-5, 0, 3, -2, 8, -1],  "with negatives"),
    (list(range(20, 0, -1)), "reverse 1-20"),
]
print(f"{'Input':<34} {'Sorted':<34} {'Shifts':>6}  {'Case'}")
print("-" * 95)
for data, label in test_cases:
    result, shifts, _ = shell_sort(data)
    print(f"{str(data):<34} {str(result):<34} {shifts:>6}  {label}")

---
## PART 3 — Searching Algorithm: Sentinel Linear Search

**Sentinel Search** eliminates the bounds check by placing the **target itself** at the end as a sentinel, leaving only **one comparison per iteration** instead of two.

```
Plain:    while i < n AND arr[i] != target   (2 checks)
Sentinel: arr[n] = target
          while arr[i] != target             (1 check)
          if i < n: found else: not found
```

In [ ]:
def sentinel_search(arr, target, verbose=False):
    if not arr:
        return -1
    a = arr[:]
    n = len(a)
    a.append(target)
    i = 0
    while a[i] != target:
        if verbose:
            print(f"  index {i}: {a[i]} ≠ {target} — continue")
        i += 1
    if verbose:
        loc = f"index {i}" if i < n else "sentinel position"
        print(f"  index {i}: {a[i]} = {target} — STOP ({loc})")
    return i if i < n else -1

In [ ]:
data = [4, 2, 7, 1, 9, 3, 6]
print(f"Array: {data}\n")
print("Search for 9 (present):")
idx = sentinel_search(data, 9, verbose=True)
print(f"Result: index {idx}\n")
print("Search for 5 (absent):")
idx = sentinel_search(data, 5, verbose=True)
print(f"Result: {idx} (not found)")

In [ ]:
search_tests = [
    ([4, 2, 7, 1, 9, 3, 6], 9,   "found in middle"),
    ([4, 2, 7, 1, 9, 3, 6], 4,   "found at start"),
    ([4, 2, 7, 1, 9, 3, 6], 6,   "found at end"),
    ([4, 2, 7, 1, 9, 3, 6], 5,   "not found"),
    ([42],                   42,  "single element — found"),
    ([42],                   1,   "single element — not found"),
    ([],                     5,   "empty list"),
    ([1, 1, 1, 1],           1,   "all same — returns first"),
]
print(f"{'Array':<30} {'Target':>7}  {'Result':<12} {'Case'}")
print("-" * 72)
for arr, target, label in search_tests:
    result = sentinel_search(arr, target)
    found  = f"index {result}" if result != -1 else "not found"
    print(f"{str(arr):<30} {str(target):>7}  {found:<12} {label}")

---
## PART 4 — End-to-End: RPN Evaluator · Shell Sort · Sentinel Search

RPN (postfix) uses a Stack to evaluate expressions without parentheses:
```
(3+4)*2  →  3 4 + 2 *  →  push 3, push 4, pop+add=7, push 2, pop*=14
```

In [ ]:
def eval_rpn(tokens):
    ops = {'+', '-', '*', '/'}
    stack = Stack()
    for token in tokens:
        if token in ops:
            b, a = stack.pop(), stack.pop()
            if   token == '+': stack.push(a + b)
            elif token == '-': stack.push(a - b)
            elif token == '*': stack.push(a * b)
            elif token == '/': stack.push(int(a / b))
        else:
            stack.push(int(token))
    return stack.pop()

expressions = [
    (['3','4','+','2','*'],          14, "(3+4)*2"),
    (['5','1','2','+','4','*','+','3','-'], 14, "5+((1+2)*4)-3"),
    (['2','3','4','+','*'],          14, "2*(3+4)"),
    (['10','6','9','3','+','-','*'], -30, "10*(6-(9+3))"),
]
print("1. RPN Evaluation using Stack:")
results = []
for tokens, expected, label in expressions:
    result = eval_rpn(tokens)
    results.append(result)
    ok = '✓' if result == expected else f'✗ (expected {expected})'
    print(f"   {' '.join(tokens):<35} = {result:>4}  {ok}")

print(f"\n2. RPN results unsorted: {results}")
sorted_results, shifts, gaps = shell_sort(results, gap_sequence='knuth')
print(f"   After Shell Sort (Knuth, {shifts} shifts): {sorted_results}")

target = 14
pos = sentinel_search(sorted_results, target)
print(f"\n3. Sentinel Search for {target}: index {pos} → {sorted_results[pos] if pos != -1 else 'not found'}")

---
## Complexity Cheat Sheet

```
Stack push/pop/peek      O(1) / O(1) / O(1)
Shell Sort (Shell n//2)  Best O(n log n), Worst O(n²)
Shell Sort (Knuth 3k+1)  Best O(n log n), Worst O(n^1.5)
Sentinel Linear Search   O(n), 1 comparison/iteration (vs 2 plain)
```

**Tomorrow — Day 005:** Queue · Merge Sort · Interpolation Search